# Сверка трех методов и итоговый датасет

Три метода брали таблицу Минфина из разных мест и разбирали ее разным кодом: из docx через python-docx, из HTML-страницы через BeautifulSoup и из отрисованной страницы через Playwright. Если бы в каком-то из них была ошибка разбора, она проявилась бы расхождением. Поэтому эту сверку я считаю главной проверкой проекта, сильнее проверок внутри каждого метода.

## 0. Период

Даты выплаты, за которые собираются данные, в формате ДД.ММ.ГГГГ, обе включительно. Пояснения и цифры ниже относятся к периоду 01.01.2024-27.10.2026.

In [1]:
# период по дате выплаты, обе даты включительно, формат ДД.ММ.ГГГГ
DATE_FROM = "01.01.2024"
DATE_TO = "27.10.2026"

## 1. Общие цифры трех методов совпадают

Тетрадь берет файлы методов за период из ячейки выше: период записан в их именах, и если какого-то файла нет, сверка останавливается. Сначала я сравнил грубые цифры: число строк, число сообщений и общую сумму. Номер купона читаю как целое (`Int64`): когда я впервые записал итоговый файл, он отличался от исходного, потому что pandas без явного типа сделал колонку дробной и записал `10.0` вместо `10`.

In [2]:
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from datetime import datetime
from pathlib import Path

import pandas as pd

start, end = datetime.strptime(DATE_FROM, "%d.%m.%Y"), datetime.strptime(DATE_TO, "%d.%m.%Y")
STEM = f"{start:%Y-%m-%d}_{end:%Y-%m-%d}"  # период в именах файлов
KEY = ["payment_date", "issue", "payment_type"]
VALUES = ["amount_mln_rub", "coupon_no", "coupon_rate_pct", "coupon_per_bond_rub"]
files = {name: Path(f"../{folder}/ofz_payments_{name}_{STEM}.csv")
         for name, folder in [("api", "01_api"), ("html", "02_html"), ("browser", "03_browser")]}
missing = [path.as_posix() for path in files.values() if not path.exists()]
if missing:  # сверять можно только результаты методов за один и тот же период
    raise FileNotFoundError(f"Нет результатов методов за этот период: {missing}. "
                            "Запустите тетради методов с теми же датами")
# номер купона читаем как целое: у погашений он пустой, и без этого стал бы дробным
data = {name: pd.read_csv(path, dtype={"coupon_no": "Int64"}) for name, path in files.items()}
pd.DataFrame([{"метод": name, "строк": len(df), "сообщений": df["doc_id"].nunique(),
               "сумма, млн руб.": f"{df['amount_mln_rub'].sum():,.3f}".replace(",", " ")}
              for name, df in data.items()]).set_index("метод")

,строк,сообщений,"сумма, млн руб."
метод,,,
api,449,140,12 274 293.547
html,449,140,12 274 293.547
browser,449,140,12 274 293.547


У всех трех по 449 строк, 140 сообщений и сумма 12 274 293,547 млн руб. Этого мало: одинаковые итоги могут дать и ошибки, которые компенсируют друг друга, поэтому дальше я сверял каждую строку.

## 2. Построчная сверка не находит ни одного расхождения

Здесь join как раз уместен: я соединял таблицу HTML-метода с таблицей другого метода по ключу «дата + выпуск + тип», внешним соединением, чтобы увидеть строки, которые есть только у одного. У совпавших строк сравнивал сумму, номер купона, ставку и купон на облигацию. Ссылки не сравнивал: у метода 1 они ведут на docx, у остальных на страницу. Если API совпадает с HTML и браузер совпадает с HTML, то совпадают и все три.

In [4]:
problems = {}
# основа сверки HTML-метод; outer join покажет строки, которые есть только у одного метода
for name in ["api", "browser"]:
    merged = data["html"].merge(data[name], on=KEY, how="outer",
                                 suffixes=("_html", f"_{name}"), indicator=True)
    only_one_side = int((merged["_merge"] != "both").sum())
    # пустые значения (у погашений нет купона) заменяются одной меткой, чтобы пустое совпадало с пустым
    diffs = {c: int((merged[f"{c}_html"].fillna(-1) != merged[f"{c}_{name}"].fillna(-1)).sum())
             for c in VALUES}
    problems[name] = only_one_side + sum(diffs.values())
    print(f"html vs {name}: строк только в одном = {only_one_side}, различий по значениям = {diffs}")

html vs api: строк только в одном = 0, различий по значениям = {'amount_mln_rub': 0, 'coupon_no': 0, 'coupon_rate_pct': 0, 'coupon_per_bond_rub': 0}
html vs browser: строк только в одном = 0, различий по значениям = {'amount_mln_rub': 0, 'coupon_no': 0, 'coupon_rate_pct': 0, 'coupon_per_bond_rub': 0}


В обеих парах 0 строк только у одного метода и 0 различий в значениях.

## 3. Итоговый файл записывается только при полном совпадении

Итоговый файл `data_files/ofz_payments_<с>_<по>.csv` тетрадь пишет, только если обе сверки дали ноль, иначе останавливается с ошибкой. Содержимое берется из HTML-метода, но раз таблицы совпали, это не важно: побайтно файл совпадает с файлом HTML-метода за тот же период. За 01.01.2024-27.10.2026 он совпадает и с `ofz_payments_2024_2026.csv` в корне репозитория, итогом первого сбора.

In [5]:
# итоговый файл пишется только при полном совпадении
if any(problems.values()):
    raise ValueError(f"Методы расходятся: {problems}. Итоговый файл не записан")
final = data["html"]
out = Path("../data_files") / f"ofz_payments_{STEM}.csv"
out.parent.mkdir(exist_ok=True)
final.to_csv(out, index=False, encoding="utf-8")
print("Методы совпали. Записан data_files/" + out.name + ":", len(final), "строк")
final.groupby("payment_type")["amount_mln_rub"].agg(["count", "sum"]).round(3)

Методы совпали. Записан data_files/ofz_payments_2024-01-01_2026-10-27.csv: 449 строк


,count,sum
payment_type,,
купон,425,8158196.121
погашение,18,4027586.379
частичное погашение,6,88511.047


Итог: 425 купонов, 18 погашений и 6 частичных погашений.